# Topic 4: Stateful Aggregations

> **Phase 6 → Week 10 → Topic 4**

---

## What is State in Streaming?

A stateful operation requires memory of **past rows** to compute its result. Spark stores this state in a **State Store** (RocksDB or HDFS-backed) that persists across micro-batches.

```
Batch 1: [A:3, B:2, C:5]  →  State: {A:3, B:2, C:5}    → Output (complete): A:3, B:2, C:5
Batch 2: [A:1, C:2]       →  State: {A:4, B:2, C:7}    → Output (complete): A:4, B:2, C:7
Batch 3: [B:6, D:1]       →  State: {A:4, B:8, C:7, D:1} → Output (complete): all 4 keys

Without state: each batch would give only its own partial counts
With state:    running totals are maintained across batches
```

---

## The State Store Problem: Unbounded Growth

```
groupBy("user_id").count()

Without watermark:
  Day 1:  100K unique users → state size: 100K entries
  Day 30: 3M unique users  → state size: 3M entries → OOM?

  Spark NEVER evicts state without a watermark.
  The state store grows until the job runs out of memory.

With watermark:
  Old keys (older than watermark) are evicted from state.
  → Bounded memory usage ✓
  → Only UPDATE output mode works (not complete)
```

---

## Interview Cheat Sheet

**Q: What is the Spark State Store?**
> The State Store is a key-value store maintained by Spark for stateful streaming operations. It holds intermediate aggregation results (e.g., running counts per group) across micro-batches. Spark checkpoints the state store to HDFS/S3 so state survives restarts. By default it is an in-memory store backed by HDFS. Spark 3.2+ introduced RocksDB-based state store for better performance with large state.

**Q: Why does groupBy().count() without watermark lead to OOM?**
> Without a watermark, Spark must retain state for every group key forever — it cannot know if an old key will receive late data. Over time the state store grows unboundedly. The fix is either (a) add a watermark to evict old state, or (b) use complete output mode only for small, finite group cardinalities.

**Q: What output mode do you use for streaming aggregations?**
> - `complete`: rewrite entire result table every trigger. Works without watermark. Use only for low-cardinality groupBys.
> - `update`: write only rows that changed. Requires watermark for memory safety. Most efficient for production aggregations.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time

spark = SparkSession.builder \
    .appName("Week10 - Stateful Aggregations") \
    .master("local[4]") \
    .config("spark.sql.shuffle.partitions", "4") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print("Spark ready")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/05/16 08:50:46 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark ready


## Part 1: Running Count per Group (complete mode)

In [2]:
# Running count per category — complete mode (full result every trigger)
stream = spark.readStream \
    .format("rate") \
    .option("rowsPerSecond", 20) \
    .load() \
    .withColumn("category", F.element_at(
        F.array(F.lit("electronics"), F.lit("clothing"), F.lit("food"), F.lit("sports")),
        (F.col("value") % 4 + 1).cast("int"))) \
    .withColumn("amount", (F.col("value") % 200 + 10).cast("double"))

agg_stream = stream.groupBy("category").agg(
    F.count("*").alias("order_count"),
    F.round(F.sum("amount"), 2).alias("total_revenue"),
    F.round(F.avg("amount"), 2).alias("avg_order_value"),
)

q = agg_stream.writeStream \
    .format("memory") \
    .queryName("category_agg") \
    .outputMode("complete") \
    .trigger(processingTime="2 seconds") \
    .start()

for i in range(4):
    time.sleep(3)
    print(f"--- t={(i+1)*3}s (state accumulating) ---")
    spark.sql("SELECT * FROM category_agg ORDER BY category").show()

q.stop()
print("Note: counts grew each snapshot — state persisted across micro-batches")

26/05/16 08:50:52 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-3944b884-0a97-4283-8483-131fb4cb9080. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/05/16 08:50:52 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


--- t=3s (state accumulating) ---


+--------+-----------+-------------+---------------+
|category|order_count|total_revenue|avg_order_value|
+--------+-----------+-------------+---------------+
+--------+-----------+-------------+---------------+



26/05/16 08:50:59 WARN ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 2000 milliseconds, but spent 5172 milliseconds


--- t=6s (state accumulating) ---


...

--- t=9s (state accumulating) ---


+-----------+-----------+-------------+---------------+
|   category|order_count|total_revenue|avg_order_value|
+-----------+-----------+-------------+---------------+
|   clothing|         25|       1475.0|           59.0|
|electronics|         25|       1450.0|           58.0|
|       food|         25|       1500.0|           60.0|
|     sports|         25|       1525.0|           61.0|
+-----------+-----------+-------------+---------------+



--- t=12s (state accumulating) ---


+-----------+-----------+-------------+---------------+
|   category|order_count|total_revenue|avg_order_value|
+-----------+-----------+-------------+---------------+
|   clothing|         60|       5740.0|          95.67|
|electronics|         60|       5680.0|          94.67|
|       food|         60|       5800.0|          96.67|
|     sports|         60|       5860.0|          97.67|
+-----------+-----------+-------------+---------------+

Note: counts grew each snapshot — state persisted across micro-batches


26/05/16 08:51:08 ERROR TorrentBroadcast: Store broadcast broadcast_21 fail, remove all pieces of the broadcast


## Part 2: Multiple Aggregations

In [3]:
# Running stats per customer
customer_stream = spark.readStream \
    .format("rate") \
    .option("rowsPerSecond", 15) \
    .load() \
    .withColumn("customer_id", F.concat(F.lit("C"), (F.col("value") % 5 + 1).cast("string"))) \
    .withColumn("amount",      (F.col("value") % 500 + 1).cast("double")) \
    .withColumn("is_premium",  (F.col("value") % 3 == 0))

customer_stats = customer_stream.groupBy("customer_id").agg(
    F.count("*").alias("total_orders"),
    F.sum("amount").alias("lifetime_value"),
    F.max("amount").alias("largest_order"),
    F.min("amount").alias("smallest_order"),
    F.sum(F.col("is_premium").cast("int")).alias("premium_orders"),
)

q2 = customer_stats.writeStream \
    .format("memory") \
    .queryName("customer_stats") \
    .outputMode("complete") \
    .trigger(processingTime="2 seconds") \
    .start()

time.sleep(10)
print("Customer lifetime stats (stateful — built up across batches):")
spark.sql("""
    SELECT customer_id, total_orders,
           round(lifetime_value, 0) as lifetime_value,
           largest_order, premium_orders
    FROM customer_stats
    ORDER BY customer_id
""").show()

q2.stop()

26/05/16 08:51:08 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-0c9e5c83-6aab-4475-b68f-35a5553e39bb. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/05/16 08:51:08 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


Customer lifetime stats (stateful — built up across batches):


+-----------+------------+--------------+-------------+--------------+
|customer_id|total_orders|lifetime_value|largest_order|premium_orders|
+-----------+------------+--------------+-------------+--------------+
|         C1|          27|        1782.0|        131.0|             9|
|         C2|          27|        1809.0|        132.0|             9|
|         C3|          27|        1836.0|        133.0|             9|
|         C4|          27|        1863.0|        134.0|             9|
|         C5|          27|        1890.0|        135.0|             9|
+-----------+------------+--------------+-------------+--------------+



## Part 3: Streaming Deduplication

In [4]:
# dropDuplicates() on a stream maintains a state of all seen keys
# Each incoming row is checked against state — if already seen, dropped

# Simulate a stream where event_id repeats (duplicates from at-least-once source)
dup_stream = spark.readStream \
    .format("rate") \
    .option("rowsPerSecond", 10) \
    .load() \
    .withColumn("event_id", (F.col("value") % 20).cast("string")) \
    .withColumn("payload",  F.concat(F.lit("data_"), F.col("value").cast("string")))

deduped = dup_stream.dropDuplicates(["event_id"])

q3 = deduped.writeStream \
    .format("memory") \
    .queryName("deduped_events") \
    .outputMode("append") \
    .trigger(processingTime="2 seconds") \
    .start()

time.sleep(10)
count = spark.sql("SELECT count(*) as unique_events FROM deduped_events").collect()[0][0]
print(f"Unique events after dedup: {count} (should be ≤ 20, one per event_id 0-19)")
spark.sql("SELECT event_id, count(*) as cnt FROM deduped_events GROUP BY event_id ORDER BY CAST(event_id AS INT)").show(25)

q3.stop()
print("Each event_id appears exactly once — duplicates dropped via state store")

26/05/16 08:51:19 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-bba4673e-e038-4370-997b-4ee3a81875a9. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/05/16 08:51:19 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


Unique events after dedup: 20 (should be ≤ 20, one per event_id 0-19)


+--------+---+
|event_id|cnt|
+--------+---+
|       0|  1|
|       1|  1|
|       2|  1|
|       3|  1|
|       4|  1|
|       5|  1|
|       6|  1|
|       7|  1|
|       8|  1|
|       9|  1|
|      10|  1|
|      11|  1|
|      12|  1|
|      13|  1|
|      14|  1|
|      15|  1|
|      16|  1|
|      17|  1|
|      18|  1|
|      19|  1|
+--------+---+

Each event_id appears exactly once — duplicates dropped via state store


26/05/16 08:51:30 ERROR WriteToDataSourceV2Exec: Data source write support MicroBatchWrite[epoch: 5, writer: org.apache.spark.sql.execution.streaming.sources.MemoryStreamingWrite@4912c239] is aborting.
26/05/16 08:51:30 ERROR WriteToDataSourceV2Exec: Data source write support MicroBatchWrite[epoch: 5, writer: org.apache.spark.sql.execution.streaming.sources.MemoryStreamingWrite@4912c239] aborted.


## Part 4: State Store — What's Happening Internally

In [5]:
print("""
State Store Internals:
════════════════════════════════════════════════════════════════

Location: spark.sql.streaming.stateStore.providerClass
  Default: HDFSBackedStateStore (in-memory with HDFS checkpoint)
  Spark 3.2+: RocksDBStateStore (better for large state, disk-spill capable)

Enable RocksDB:
  spark.conf.set(
    "spark.sql.streaming.stateStore.providerClass",
    "org.apache.spark.sql.execution.streaming.state.RocksDBStateStoreProvider"
  )

State per aggregation:
  groupBy("key").count()
    State key:   partition(key)
    State value: {count: Long, sum: Double, ...}

  dropDuplicates(["event_id"])
    State key:   event_id
    State value: seen=true (boolean)

State size grows with:
  - Number of unique group keys (high cardinality = big state)
  - Duration the job runs without evicting old keys

State is evicted when:
  - A watermark is set AND the group's event time is older than watermark
  - Without watermark: NEVER evicted → OOM risk

Monitor state size in Spark UI:
  Streaming tab → State Operators → stateMemoryUsedBytes
════════════════════════════════════════════════════════════════
""")


State Store Internals:
════════════════════════════════════════════════════════════════

Location: spark.sql.streaming.stateStore.providerClass
  Default: HDFSBackedStateStore (in-memory with HDFS checkpoint)
  Spark 3.2+: RocksDBStateStore (better for large state, disk-spill capable)

Enable RocksDB:
  spark.conf.set(
    "spark.sql.streaming.stateStore.providerClass",
    "org.apache.spark.sql.execution.streaming.state.RocksDBStateStoreProvider"
  )

State per aggregation:
  groupBy("key").count()
    State key:   partition(key)
    State value: {count: Long, sum: Double, ...}

  dropDuplicates(["event_id"])
    State key:   event_id
    State value: seen=true (boolean)

State size grows with:
  - Number of unique group keys (high cardinality = big state)
  - Duration the job runs without evicting old keys

State is evicted when:
  - A watermark is set AND the group's event time is older than watermark
  - Without watermark: NEVER evicted → OOM risk

Monitor state size in Spark UI:


26/05/16 08:51:30 ERROR TaskContextImpl: Error in TaskCompletionListener
java.io.FileNotFoundException: File file:/tmp/temporary-bba4673e-e038-4370-997b-4ee3a81875a9/state/0/1 does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:779)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1100)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:769)
	at org.apache.hadoop.fs.DelegateToFileSystem.getFileStatus(DelegateToFileSystem.java:128)
	at org.apache.hadoop.fs.DelegateToFileSystem.createInternal(DelegateToFileSystem.java:93)
	at org.apache.hadoop.fs.ChecksumFs$ChecksumFSOutputSummer.<init>(ChecksumFs.java:353)
	at org.apache.hadoop.fs.ChecksumFs.createInternal(ChecksumFs.java:400)
	at org.apache.hadoop.fs.AbstractFileSystem.create(AbstractFileSystem.java:626)
	at org.apache.hadoop.fs.FileContext$3.next(FileContext.java:701)
	at org.apache.hadoop.fs.FileContext$3

## Exercises

1. Create a streaming aggregation that computes `max(amount)` and `min(amount)` per `customer_id`. Use `complete` mode. After 10 seconds, query the result. What would happen to memory if this job ran for a year with millions of unique customers?
2. Use `dropDuplicates(["event_id", "event_date"])` on a stream. Why does adding `event_date` to the dedup key reduce state size vs just `dropDuplicates(["event_id"])`?
3. You have a streaming job counting events per `user_id`. After 30 days, you notice the job is using 20 GB of state store memory. What is the root cause and what is the fix?
4. Explain the difference between `complete` and `update` output modes for aggregations. In which scenario is `update` strictly better?
5. Can you chain two streaming aggregations? e.g., `groupBy("category").count()` then another `.groupBy()` on the result. Try it — what error do you get?